In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col, concat, lit, when, rand, cast

from pyspark.sql.types import *


In [0]:
raw_df = (
    spark.read.table('temp.1_bronze.user_dataset')
)

display(raw_df.limit(5))
# 0 drop duplicate
# 1. address >> add new col lat, lng
#            >> concate city + pincode address col
# 2. drop age, email null rows
# 3. replace full country name
# 4 isactive replace null with true,flase equaly distribute
# 5 drop last login
# 6 replace invalid phone numbder with null
# 7 project : make it project_status and pick latest
# 8 signup_date to valid date
# 9 make skill comma seperated
# 10 transaction to transaction amount sum

In [0]:
# 0 dropduplicate 

raw_df = raw_df.dropDuplicates(["id"])
display(raw_df.orderBy("id").limit(5))

In [0]:
# 1. address >> add new col lat, lng
#            >> concate city + pincode address col
raw_df = raw_df.withColumns({
    "lat": col("address.coordinates.lat").cast("double"),
    "lon": col("address.coordinates.lng").cast("double"),
    "address": concat(
        col("address.city"),
        lit('_'),
        col("address.pincode")
    )
})

In [0]:
# 2. drop age, email null rows

# raw_df = raw_df.drop("age", "email")
raw_df = raw_df.dropna(subset=["age", "email"])

# 3. replace full country name

raw_df = raw_df.withColumns({
    "country": when(col("country")=='UK', "United Kingdom")
    .when(col("country")=="USA", "United Atates Of America")
    .when(col("country")=="AUSTRALIA", "Australia")
    .otherwise(col("country"))
})

# display(raw_df.limit(5))



In [0]:
# 4 isactive replace null with true,flase equaly distribute

raw_df = raw_df.withColumn(
    "is_active",
    when(col("is_active").isNull(),
         when(rand() > 0.5, True).otherwise(False)
    ).otherwise(col("is_active"))
)

# display(raw_df.limit(5))

In [0]:

# 5 drop last login

raw_df = raw_df.drop("last_login")
# display(raw_df.limit(5))


In [0]:
# 6 replace invalid phone numbder with null

raw_df = raw_df.withColumn(
    "phone",
    when(col("phone").rlike("^[0-9]{10}$"), col("phone"))  # valid
    .otherwise(None)  # invalid → NULL
)
# display(raw_df.limit(5))



In [0]:
# 7 project : make it project_status and pick latest

from pyspark.sql.functions import col, when, array, array_contains

raw_df = raw_df.withColumn(
    "project_status",
    when(array_contains(array(
        col("projects.project_0"),
        col("projects.project_1"),
        col("projects.project_2")
    ), "completed"), "completed")
    
    .when(array_contains(array(
        col("projects.project_0"),
        col("projects.project_1"),
        col("projects.project_2")
    ), "in_progress"), "in_progress")
    
    .when(array_contains(array(
        col("projects.project_0"),
        col("projects.project_1"),
        col("projects.project_2")
    ), "failed"), "failed")
    
    .otherwise(None) 
)

# display(raw_df.limit(5))

In [0]:
# 8 signup_date to valid date


from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, DateType
from dateutil import parser

def parse_date_safe(date_str):
    try:
        if date_str is None:
            return None
        
        # Parse date automatically
        dt = parser.parse(date_str, dayfirst=True)  # important for 13/02/2024
        
        # Convert to desired format
        # return dt.strftime("%d/%m/%Y")
        return dt.date()
    
    except:
        return None  # invalid → NULL

parse_udf = udf(parse_date_safe, DateType())

raw_df = raw_df.withColumn("signup_date", parse_udf("signup_date"))
display(raw_df.limit(5))


In [0]:
# 9 make skill comma seperated

from pyspark.sql.functions import upper, transform, trim, array_distinct, concat_ws
# raw_df = raw_df.withColumn("skills", col("skills")).withColumn("skills",transform(col("skills"), lambda x: x.strip())).withColumn("skills",transform(col("skills"), lambda x: ', '.join(x)))

raw_df = raw_df.withColumn(
    "skills",
    transform(col("skills"), lambda x: trim(x))
)

raw_df = raw_df.withColumn(
    "skills",
    transform(col("skills"), lambda x: upper(x))
)

raw_df = raw_df.withColumn(
    "skills",
    array_distinct(col("skills"))
)

raw_df = raw_df.withColumn(
    "skills",
    concat_ws(", ", col("skills"))
)


# display(raw_df.limit(5))

In [0]:
display(raw_df.limit(5))

In [0]:
# 10 transaction to transaction amount sum
from pyspark.sql.functions import udf, expr
from pyspark.sql.types import DoubleType


def sum_amount(arr):
    if arr is None:
        return 0.0
    
    total = 0.0
    for x in arr:
        if x is not None and x.amount is not None:
            total += float(x.amount)
    
    return total
    
    return total

sum_udf = udf(sum_amount, DoubleType())

raw_df = raw_df.withColumn("total_amount", sum_udf("transactions"))  # Ensure 'transactions' is an array of structs, not a MAP

raw_df = raw_df.withColumn(
    "total_amount2",
    expr("""
        aggregate(transactions, 0D,
        (acc, x) -> acc + coalesce(x.amount, 0))
    """)
)


rows = raw_df.count()
cols = len(raw_df.columns)

print((rows, cols))

In [0]:

from delta.tables import DeltaTable

table_name = "temp.2_silver.user_dataset"

if spark.catalog.tableExists(table_name):
    delta_table = DeltaTable.forName(spark, table_name)

    delta_table.alias("t").merge(
        raw_df.alias("s"),
        "t.id = s.id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    raw_df.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("country") \
        .saveAsTable(table_name)

In [0]:
# from pyspark.sql.types import *

# schema = StructType([
#     StructField("address", StringType(), True),
#     StructField("age", IntegerType(), True),
#     StructField("country", StringType(), True),
#     StructField("email", StringType(), True),
#     StructField("id", IntegerType(), True),
#     StructField("is_active", BooleanType(), True),
#     StructField("name", StringType(), True),
#     StructField("phone", StringType(), True),
#     StructField("projects", StringType(), True),
#     StructField("salary", DoubleType(), True),
#     StructField("signup_date", DateType(), True),
#     StructField("skills", StringType(), True),
#     StructField("transactions", ArrayType(
#         StructType([
#                 StructField("amount", DoubleType(), True),
#                 StructField("date", StringType(), True),
#                 StructField("txn_id", StringType(), True)
#             ])
#         ), True),
#     StructField("ingetiontime", TimestampType(), True), 
#     StructField("filename", StringType(), True),
#     StructField("lat", DoubleType(), True), 
#     StructField("lon", DoubleType(), True), 
#     StructField("project_status", StringType(), True),
#     StructField("total_amount", DoubleType(), True),
#     StructField("total_amount2", DoubleType(), True)
# ])

In [0]:
%sql
-- select * from temp.2_silver.user_dataset order by id;